# Bacterial Growth: Linear Regression with a Log Transformation
Adapted from the supplied bacteria-doubling code to **growth-OD600-Blank_correct.xlsx**.

This workbook contains measured blank-corrected OD600 readings, rather than simulated population counts. We do not convert these readings into numbers of bacteria. Time is the single input; OD600 is the target. The workbook has 96 wells and 226 measurement times, including four blank wells with no readings.

Run the cells from top to bottom. Keep the spreadsheet beside this notebook. This is a teaching analysis of one well, not a comparison of antibiotic treatments.

## 1. Setup

In [ ]:
# Uncomment only if packages are missing, then restart the kernel.
# %pip install numpy pandas matplotlib scikit-learn openpyxl
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

## 2. Read the actual Excel layout
The first row describes the columns. The second row contains time in hours. Each remaining row is one well, with its treatment in the third column. We reshape the readings into one row per well and time. The values are already blank corrected; we do not subtract the blank again.

In [ ]:
DATA_FILE = Path("growth-OD600-Blank_correct.xlsx")
if not DATA_FILE.exists():
    DATA_FILE = Path("../upload/growth-OD600-Blank_correct.xlsx")
raw = pd.read_excel(DATA_FILE, sheet_name="All Cycles", header=None)
times = pd.to_numeric(raw.iloc[1, 3:], errors="coerce").to_numpy()
metadata = raw.iloc[2:, :3].copy()
metadata.columns = ["well_row", "well_col", "treatment"]
metadata["well"] = metadata["well_row"].astype(str) + metadata["well_col"].astype(int).astype(str)
readings = raw.iloc[2:, 3:].apply(pd.to_numeric, errors="coerce")
readings.columns = times
wide = pd.concat([metadata[["well", "treatment"]], readings], axis=1)
long_df = wide.melt(id_vars=["well", "treatment"], var_name="time_hours", value_name="od600")
long_df["time_hours"] = pd.to_numeric(long_df["time_hours"], errors="coerce")
print("Wells:", metadata["well"].nunique(), "Measurement times:", len(times))
print("Missing OD600 readings:", long_df["od600"].isna().sum())
display(metadata[["well", "treatment"]].head(12))
display(long_df.head())

## 3. Choose one well and inspect the full curve
Default: **E12**, labelled **Positive control P**. Keeping one well avoids mixing different treatments into a single time-only model. Change `SELECTED_WELL` to inspect another well, then review its growth window.

The editable **3–6 hour** interval is an illustrative rising segment chosen after inspecting this curve. It is not an independently validated exponential phase. Using the full curve to choose the interval makes subsequent test scores exploratory.

In [ ]:
SELECTED_WELL = "E12"
START_HOUR = 3.0
END_HOUR = 6.0
selected = long_df.loc[long_df["well"] == SELECTED_WELL].copy()
if selected.empty:
    raise ValueError("Unknown well. Choose a well from metadata.")
treatment = selected["treatment"].iloc[0]
df = selected.dropna(subset=["time_hours", "od600"]).sort_values("time_hours").reset_index(drop=True)
if df.empty:
    raise ValueError("This well has no readings; choose a nonblank well.")
print("Selected:", SELECTED_WELL, treatment)
print("Nonpositive readings:", (df["od600"] <= 0).sum())
display(df[["time_hours", "od600"]].describe().round(4))
plt.scatter(df["time_hours"], df["od600"], s=12, label="Measured OD600")
plt.axvspan(START_HOUR, END_HOUR, alpha=0.18, color="orange", label="Chosen modelling window")
plt.xlabel("Time (hours)"); plt.ylabel("Blank-corrected OD600")
plt.title(f"Full growth curve: {SELECTED_WELL} — {treatment}")
plt.legend(); plt.show()

## 4. Prepare the growth window and transform the target
`np.log()` requires strictly positive values. Blank correction can leave zero or negative readings. We report and exclude them from **both** models within the selected window; we do not replace them with an arbitrary small number.

For an exponential relationship, `OD600 = a × exp(b × time)`, taking the logarithm gives `ln(OD600) = ln(a) + b × time`. A log transformation does not remove lag, plateau, or decline phases.

In [ ]:
window = df.loc[df["time_hours"].between(START_HOUR, END_HOUR)].copy()
print("Window readings:", len(window))
print("Excluded nonpositive readings in window:", (window["od600"] <= 0).sum())
model_df = window.loc[window["od600"] > 0].copy()
if len(model_df) < 8:
    raise ValueError("Choose a window with at least 8 positive readings.")
model_df["log_od600"] = np.log(model_df["od600"])
display(model_df[["time_hours", "od600", "log_od600"]].head())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(model_df["time_hours"], model_df["od600"])
axes[0].set(title="Original target", xlabel="Time (hours)", ylabel="OD600")
axes[1].scatter(model_df["time_hours"], model_df["log_od600"])
axes[1].set(title="Log-transformed target", xlabel="Time (hours)", ylabel="ln(OD600)")
plt.tight_layout(); plt.show()

## 5. Use the same train/test split
Reserve 25% of the selected positive readings. This random split assesses held-out points within the chosen interval. Nearby time readings are related, so this is not independent experimental validation or evidence of future forecasting performance.

In [ ]:
X = model_df[["time_hours"]]
y = model_df["od600"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Training readings:", len(X_train), "Test readings:", len(X_test))

## 6. Train a direct model and a log-target model

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)
log_model = LinearRegression()
log_model.fit(X_train, np.log(y_train))
log_predictions = log_model.predict(X_test)
exponential_predictions = np.exp(log_predictions)
estimated_a = np.exp(log_model.intercept_)
estimated_b = log_model.coef_[0]
print(f"Direct fit: OD600 = {linear_model.intercept_:.5f} + {linear_model.coef_[0]:.5f} × time")
print(f"Log fit: ln(OD600) = {log_model.intercept_:.5f} + {estimated_b:.5f} × time")
print(f"Back-transformed fit: OD600 = {estimated_a:.5f} × exp({estimated_b:.5f} × time)")
if estimated_b > 0:
    print(f"Fitted OD600 doubling interval: {np.log(2) / estimated_b:.3f} hours")

The doubling interval describes this fitted OD600 curve within the chosen window; it is not a directly measured cell-population doubling time. The parameter `a` extrapolates the selected-window curve to time zero, where this fit may not apply. Exponentiating a log prediction also does not automatically give an unbiased original-scale mean.

## 7. Compare both models in OD600 units
Lower RMSE means smaller errors. R² compares predictions with using the test mean; it can be negative and is not a percentage accuracy. Compare both models on the original target, not one on OD600 and the other on ln(OD600).

In [ ]:
results = []
for name, predictions in [("Direct linear regression", linear_predictions), ("Log-target regression", exponential_predictions)]:
    results.append({"Model": name, "Test RMSE (OD600)": np.sqrt(mean_squared_error(y_test, predictions)), "Test R²": r2_score(y_test, predictions)})
metrics = pd.DataFrame(results).set_index("Model")
display(metrics.round(5))
comparison = pd.DataFrame({"Time (hours)": X_test["time_hours"], "Actual OD600": y_test,
                           "Direct prediction": linear_predictions, "Log-model prediction": exponential_predictions}).sort_values("Time (hours)")
display(comparison.round(5))
print("Lower test RMSE in this run:", metrics["Test RMSE (OD600)"].idxmin())
time_grid = pd.DataFrame({"time_hours": np.linspace(model_df["time_hours"].min(), model_df["time_hours"].max(), 200)})
plt.scatter(X_train["time_hours"], y_train, alpha=0.6, label="Training readings")
plt.scatter(X_test["time_hours"], y_test, marker="x", s=70, label="Test readings")
plt.plot(time_grid["time_hours"], linear_model.predict(time_grid), color="crimson", label="Direct linear fit")
plt.plot(time_grid["time_hours"], np.exp(log_model.predict(time_grid)), color="green", label="Log-target fit, back transformed")
plt.xlabel("Time (hours)"); plt.ylabel("OD600")
plt.title(f"Model comparison within selected window: {SELECTED_WELL}")
plt.legend(); plt.show()

## 8. Residuals: actual minus predicted
Positive residuals mean underprediction; negative residuals mean overprediction. Both panels use OD600 units and the same test readings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, title, predictions in [(axes[0], "Direct linear residuals", linear_predictions), (axes[1], "Log-target model residuals", exponential_predictions)]:
    ax.scatter(X_test["time_hours"], y_test.to_numpy() - predictions)
    ax.axhline(0, color="black", linestyle="--")
    ax.set(title=title, xlabel="Time (hours)")
axes[0].set_ylabel("Actual − predicted (OD600)")
plt.tight_layout(); plt.show()

## 9. Three talking points for your walkthrough
1. **Preparing:** “Our Excel file stores time across columns and wells across rows. I reshaped it, selected positive-control well E12, and used time to predict blank-corrected OD600.”
2. **Transforming:** “The full experiment includes a plateau, so I chose an illustrative rising interval. I fitted one model directly to positive OD600 readings and another to their natural logarithms, then used `np.exp()` to convert predictions back.”
3. **Evaluating:** “I compared RMSE and R² on the same test readings in OD600 units. The results apply to this selected interval and well, not all treatments or future growth.”

### Practice questions
- Why do we use two brackets in `df[["time_hours"]]`?
- Why must the readings be positive before taking a logarithm?
- Why is `np.exp()` needed after log-target prediction?
- Why should we avoid applying the fitted exponential curve to the plateau?
- Does the measured result support the transformation in this interval?

### Conclusion to complete
Use the displayed metrics and plots to state which model fits this interval better. Explain the remaining residual pattern. Do not assume that the log transformation must win. Review the window before changing wells; do not choose windows merely to obtain the highest test score.